<a href="https://colab.research.google.com/github/K-sew/Converter/blob/main/%D0%92%D1%82%D0%BE%D1%80%D0%BE%D0%B9_%D0%BA%D0%B5%D0%B9%D1%81.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 2 кейс

**ЗАДАНИЕ**

У онлайн-школы есть множество студентов. У некоторых из них оформлены рассрочки на обучение, и нужно следить за тем, чтобы все студенты вовремя оплачивали обучение.
Вся информация о студентах, их платежах, включая даты последних полученных и ожидаемых платежей собрана в Google Sheets.

Необходимо собрать информацию из **Google Sheets**, обработать ее и сгенерировать отчет, в котором будет указано, у каких студентов есть долг.

Задача - написать скрипт, который сохранит результаты "анализа" в текстовый файл формата **"Студент Иванов И.И. - долг N рублей"**. Оформить скрипт в функцию **generate_report**, которая принимает три списка (**данные с каждого листа в Google Sheets**) и сохраняет отчет в файл **student_debt_report.txt**.

Важно: иногда встречаются злостные неплательщики, которые пропустили уже несколько платежей - это тоже нужно обязательно отразить в отчете. То есть их долг должен быть больше, чем простой разовый платеж.

**Важно**

Перед началом решения задачи выполните следующую ячейку - в ней скачиваются нужные файлы

In [ ]:
!wget https://gist.github.com/Vs8th/d0bd4bdbbb58c8ae4f70a2a503e2d5fc/raw/creds.json

!wget https://gist.github.com/Vs8th/39c5deed0f5539d781f00328f7fd4fe0/raw/result.txt

--2026-08-11 12:17:39--  https://gist.github.com/Vs8th/d0bd4bdbbb58c8ae4f70a2a503e2d5fc/raw/creds.json
Resolving gist.github.com (gist.github.com)... 140.82.114.3
Connecting to gist.github.com (gist.github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/d0bd4bdbbb58c8ae4f70a2a503e2d5fc/raw/creds.json [following]
--2026-08-11 12:17:39--  https://gist.githubusercontent.com/Vs8th/d0bd4bdbbb58c8ae4f70a2a503e2d5fc/raw/creds.json
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2358 (2.3K) [text/plain]
Saving to: ‘creds.json.3’

creds.json.3        100%[===================>]   2.30K  --.-KB/s    in 0s      

2026-08-11 12:17:39 (38.4 MB/s) - ‘creds.json.3’ save

Чтобы посмотреть как выглядят сами гугл таблицы в оригинале - можете перейти по [ссылке](https://docs.google.com/spreadsheets/d/1hRnw-PEftF0J-6KY7InlILVwWdkJu4vJiGwRjuE_P4U/edit?usp=sharing) и изучить.

Небольшое описание столбцов в таблицах:  
**Лист1:** `student_id` - айди студента; `student_name` - имя студента; `installment` (Y/N) - факт наличия рассрочки у студента (Y - рассрочка есть, N - рассрочки нет).  
**Лист2:** `student_id` - айди студента; `last_payment_date` - дата последней полученной оплаты; `expected_payment_date` - дата, в которую ожидаем следующий платеж (рассчитывается от даты последнего платежа).  
**Лист3:** `student_id` - айди студента; `already_payed_amount` - уже выплаченная часть рассрочки; `left_to_pay` - оставшаяся (невыплаченная) часть; `one-time_payment` - единоразовый платеж; `installment_amount` - общая сумма, которая бралась в рассрочку.

Как примерно должен выглядеть результат:

In [ ]:
with open('result.txt') as f:
  print(f.read())

In [ ]:
#@title Если возникнут трудности с определением `scope`, подсказка:

scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
         'https://www.googleapis.com/auth/drive']

**Примечание**

Считать должников необходимо на 1 марта 2023 года. То есть определяя количество просроченных платежей, мы определяем их количество не на настоящую дату, а на **1 марта 2023 года**. А периоды внесения платежей для всех студентов одинаковы - **6 месяцев** (183 дня).

То есть, если ожидаемый платеж должен был пройти 3 февраля 2022 года, то к 1 марта 2023 студент просрочил уже три платежа (3 февраля 2022, 3 августа 2022 и 3 февраля 2023). При расчетах отталкивайтесь от даты ожидаемого платежа, разницу между платежами берите 183 дня.

**Решение**

Напишите свое решение ниже

In [ ]:
# Прежде всего подключаю google sheets api
import gspread
from oauth2client.service_account import ServiceAccountCredentials

scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
         'https://www.googleapis.com/auth/drive']

creds = ServiceAccountCredentials.from_json_keyfile_name('creds.json', scope)
client = gspread.authorize(creds)

# Далее, собираю данные с листов в переменные sheet1, sheet2, sheet3
sheet = client.open("Installments").worksheet("Лист1")
sheet1 = sheet.get_all_records()

sheet = client.open("Installments").worksheet("Лист2")
sheet2 = sheet.get_all_records()

sheet = client.open("Installments").worksheet("Лист3")
sheet3 = sheet.get_all_records()

# Создаю саму функцию
def generate_report(sheet1, sheet2, sheet3):

  # Импортирую модуль pandas и создаю датафреймы по каждому листу
  import pandas as pd
  df1 = pd.DataFrame(sheet1)
  df2 = pd.DataFrame(sheet2)
  df3 = pd.DataFrame(sheet3)

  # В таблице df2 присутствуют поля с датами типа object, перевожу их в datetime
  df2['last_payment_date'] = pd.to_datetime(df2['last_payment_date'], format='%d.%m.%Y')
  df2['expected_payment_date'] = pd.to_datetime(df2['expected_payment_date'], format='%d.%m.%Y')

  # Мне понадобится столбец 'one-time_payment', на всякий случай переведу его в  числовой формат
  df3['one-time_payment'] = pd.to_numeric(df3['one-time_payment'])

  # Теперь можно объединить таблицы:
  # Cперва объединяю первые 2 таблицы
  # В первой таблице есть студенты, которые не брали рассрочку,
  # их не беру в результирующую таблицу, используя inner join
  # Таблица со студентами, взявшими рассрочку:
  df_installment = pd.merge(df1, df2, how='inner', on='student_id')

  # Объединяю c 3 таблицей (можно inner можно Left, здесь результат будет одинак):
  df = pd.merge(df_installment, df3, how='inner', on='student_id')

  # оставлю только нужные столбцы, включая left_to_pay для проверки
  df = df[['student_id', 'student_name', 'last_payment_date', 'expected_payment_date', 'one-time_payment', 'left_to_pay']]

  # Создам переменную now, которая будет хранить дату 1 марта 2023
  now = pd.to_datetime('2023-03-01')

  # Отфильтрую тех студентов, которые являются должниками в отдельную таблицу,
  # для этого сравню значение поля expected_payment_date с now
  df_debt = df[df['expected_payment_date'] < now].copy()

  # Посчитаю сумму просрочки, для этого создам колонку debt
  # ((now - expected_payment_date) / 183 + 1) * one-time_payment
  df_debt['debt'] = ((now - df_debt.expected_payment_date).dt.days // 183 + 1) * df_debt['one-time_payment']

  # Нужно также убедиться, что долг не превышает сумму остатка
  df_debt['debt'] = df_debt[['debt', 'left_to_pay']].min(axis=1)

  # Теперь можно записывать данные из таблицы df_debt в текстовый файл
  with open('student_debt_report.txt', 'w') as f:
    for row in df_debt.itertuples():
      f.write(f'Студент {row.student_name} - долг {row.debt} рублей\n')

✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [ ]:
# Здесь будет скачиваться файл с эталонным ответом

!wget https://gist.github.com/Vs8th/63832f093f4db8d8b251ba5d39571f3d/raw/student_debt.txt

import pandas as pd

user_answer = pd.read_csv('student_debt_report.txt')
correct_answer = pd.read_csv('student_debt.txt')

--2026-08-11 12:18:00--  https://gist.github.com/Vs8th/63832f093f4db8d8b251ba5d39571f3d/raw/student_debt.txt
Resolving gist.github.com (gist.github.com)... 140.82.113.3
Connecting to gist.github.com (gist.github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/63832f093f4db8d8b251ba5d39571f3d/raw/student_debt.txt [following]
--2026-08-11 12:18:00--  https://gist.githubusercontent.com/Vs8th/63832f093f4db8d8b251ba5d39571f3d/raw/student_debt.txt
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 11325 (11K) [text/plain]
Saving to: ‘student_debt.txt.2’

student_debt.txt.2  100%[===================>]  11.06K  --.-KB/s    in 0s      

2026-08-11 12:18:01 (88.9 MB/

In [ ]:
try:
  assert (user_answer == correct_answer).all().all(), 'Ответы не совпадают'
  assert user_answer.columns.equals(correct_answer.columns), 'Названия столбцов не совпадают'
except Exception as err:
  raise AssertionError(f'При проверке возникла ошибка {repr(err)}')
else:
  print('Поздравляем, Вы справились и успешно прошли все проверки!!')

Поздравляем, Вы справились и успешно прошли все проверки!!
